# SentinelMail — Bi-LSTM + char-CNN: Training & Evaluation

Trains `models/bilstm/BiLSTMCharCNN` on the ai4privacy training split and evaluates
it on the validation split. Saves metrics to `evaluation/results/bilstm_metrics.json`
for ensemble comparison.

**Prerequisites**
- GloVe 6B 100d at `data/embeddings/glove.6B.100d.txt`  
  Download: `wget https://nlp.stanford.edu/data/glove.6B.zip && unzip glove.6B.zip -d data/embeddings/`
- PyTorch installed (`pip install torch`)

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

CHECKPOINT_DIR = REPO_ROOT / "models" / "bilstm" / "checkpoint"
GLOVE_PATH     = REPO_ROOT / "data" / "embeddings" / "glove.6B.100d.txt"
RESULTS_DIR    = REPO_ROOT / "evaluation" / "results"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"GloVe available: {GLOVE_PATH.exists()}")

In [ ]:
from data.preprocessing import load_split, LABEL_COLS
from models.bilstm.vocab   import Vocabulary
from models.bilstm.dataset import DLPTextDataset, collate_fn
from models.bilstm.model   import BiLSTMCharCNN
from models.bilstm.predict import BiLSTMDLPDetector
from evaluation import (
    compute_all_metrics,
    classify_errors,
    error_summary,
    plot_confusion_matrices,
    plot_per_label_bars,
    plot_roc_curves,
    plot_pr_curves,
    save_results,
    from_dataframe,
    LABEL_COLS,
)

## 1. Load Data

In [ ]:
print("Loading splits...")
train_df = load_split("train")
val_df   = load_split("validation")

train_texts  = train_df["source_text"].tolist()
val_texts    = val_df["source_text"].tolist()
train_labels = train_df[LABEL_COLS].values.astype(np.float32)
val_labels   = val_df[LABEL_COLS].values.astype(np.float32)

print(f"Train: {len(train_texts):,} samples")
print(f"Val:   {len(val_texts):,} samples")
print()
print("Label distribution (train):")
print(train_df[LABEL_COLS].sum().to_string())

## 2. Build Vocabulary

In [ ]:
VOCAB_PATH = CHECKPOINT_DIR / "vocab.json"

if VOCAB_PATH.exists():
    vocab = Vocabulary.load(VOCAB_PATH)
    print(f"Loaded existing vocabulary from {VOCAB_PATH}")
else:
    vocab = Vocabulary()
    vocab.build_from_texts(train_texts, min_freq=2)
    print("Built vocabulary from training data")

print(f"Words: {vocab.n_words:,}  |  Chars: {vocab.n_chars}")

## 3. Load GloVe Embeddings

In [ ]:
def load_glove(path: Path, vocab: Vocabulary, emb_dim: int = 100) -> torch.Tensor:
    emb = torch.zeros(vocab.n_words, emb_dim)
    nn.init.normal_(emb[2:], mean=0.0, std=0.01)  # PAD=0, UNK=1 stay zero
    if not path.exists():
        print("[warn] GloVe not found — using random init for all words.")
        return emb
    found = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.split()
            word = parts[0]
            if word in vocab.word2idx:
                emb[vocab.word2idx[word]] = torch.tensor([float(v) for v in parts[1:emb_dim+1]])
                found += 1
    print(f"GloVe: {found:,}/{vocab.n_words-2:,} vocab words initialised from vectors")
    return emb

pretrained_emb = load_glove(GLOVE_PATH, vocab)

## 4. Datasets & DataLoaders

In [ ]:
BATCH_SIZE = 64

train_ds = DLPTextDataset(train_texts, train_labels, vocab)
val_ds   = DLPTextDataset(val_texts,   val_labels,   vocab)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=2, pin_memory=(DEVICE.type == "cuda"),
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=2,
)

print(f"Train batches: {len(train_loader):,}  |  Val batches: {len(val_loader):,}")

## 5. Initialise Model

In [ ]:
model = BiLSTMCharCNN(vocab.n_words, vocab.n_chars, pretrained_emb=pretrained_emb).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}  (GloVe word embedding frozen)")

## 6. Train

Skip this cell if `checkpoint/best_model.pt` already exists — the next section loads it.

In [ ]:
EPOCHS = 20
LR     = 1e-3
CKPT   = CHECKPOINT_DIR / "best_model.pt"

def _pos_weights(labels: np.ndarray, device: torch.device) -> torch.Tensor:
    pos = labels.sum(axis=0).clip(1)
    neg = len(labels) - pos
    return torch.tensor(neg / pos, dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=_pos_weights(train_labels, DEVICE))
optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad], lr=LR
)

history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for word_ids, char_ids, mask, labels in tqdm(train_loader, desc=f"Epoch {epoch:02d} train", leave=False):
        word_ids, char_ids, mask, labels = (
            word_ids.to(DEVICE), char_ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
        )
        optimizer.zero_grad()
        loss = criterion(model(word_ids, char_ids, mask), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for word_ids, char_ids, mask, labels in val_loader:
            word_ids, char_ids, mask, labels = (
                word_ids.to(DEVICE), char_ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
            )
            val_loss += criterion(model(word_ids, char_ids, mask), labels).item()

    avg_train = train_loss / len(train_loader)
    avg_val   = val_loss   / len(val_loader)
    history["train_loss"].append(avg_train)
    history["val_loss"].append(avg_val)
    marker = "  ✓ saved" if avg_val < best_val_loss else ""
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), CKPT)
        vocab.save(CHECKPOINT_DIR / "vocab.json")
    print(f"Epoch {epoch:02d}/{EPOCHS}  train={avg_train:.4f}  val={avg_val:.4f}{marker}")

print(f"\nBest val loss: {best_val_loss:.4f}  →  {CKPT}")

### 6a. Learning Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
epochs_range = range(1, len(history["train_loss"]) + 1)
ax.plot(epochs_range, history["train_loss"], label="train")
ax.plot(epochs_range, history["val_loss"],   label="val")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE Loss")
ax.set_title("Bi-LSTM + char-CNN — Learning Curves")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Load Best Checkpoint

In [ ]:
CKPT     = CHECKPOINT_DIR / "best_model.pt"
VOC_PATH = CHECKPOINT_DIR / "vocab.json"

detector = BiLSTMDLPDetector(
    checkpoint_path=str(CKPT),
    vocab_path=str(VOC_PATH),
    device=str(DEVICE),
)
print(f"Loaded: {CKPT.name}")

## 8. Run Inference on Validation Set

In [ ]:
THRESHOLD = 0.5

print(f"Running predict_proba on {len(val_texts):,} samples...")
y_proba = detector.predict_proba(val_texts)       # (N, 4) float32
y_pred  = (y_proba >= THRESHOLD).astype(np.int32) # (N, 4) int
y_true  = from_dataframe(val_df)                   # (N, 4) int

print(f"y_proba range: [{y_proba.min():.3f}, {y_proba.max():.3f}]")
print()
print(f"{'label':14s}  {'pred':>6s}  {'true':>6s}")
for i, col in enumerate(LABEL_COLS):
    print(f"  {col:12s}  {y_pred[:,i].sum():6d}  {y_true[:,i].sum():6d}")

## 9. Metrics

In [ ]:
metrics = compute_all_metrics(y_true, y_pred, y_proba=y_proba)

ci = metrics["macro_f1_ci"]
print(f"Macro-F1:      {metrics['macro_f1']:.4f}  "
      f"(95% CI: [{ci['lower']:.4f}, {ci['upper']:.4f}])")
print(f"Macro AUC-ROC: {metrics['auc_roc']['macro']:.4f}")
print()

pd.DataFrame(metrics["per_label"]).T.round(4)

### 9a. Threshold Sensitivity (0.3 / 0.5 / 0.7)

In [ ]:
rows = []
for t in [0.3, 0.5, 0.7]:
    m = compute_all_metrics(y_true, (y_proba >= t).astype(np.int32), n_bootstrap=50)
    row = {"threshold": t, "macro_f1": round(m["macro_f1"], 4)}
    for label in LABEL_COLS:
        row[f"{label}_f1"] = round(m["per_label"][label]["f1"], 4)
    rows.append(row)

pd.DataFrame(rows).set_index("threshold")

## 10. Confusion Matrices & Per-Label Bars

In [ ]:
_, fig = plot_confusion_matrices(y_true, y_pred, normalize="true", return_fig=True)
fig.suptitle("Bi-LSTM + char-CNN — Confusion Matrices (val EN, threshold=0.5)", y=1.02)
plt.show()

_, bar_fig = plot_per_label_bars(metrics, return_fig=True)
bar_fig.suptitle("Bi-LSTM + char-CNN — Per-Label Metrics")
plt.show()

## 11. ROC & PR Curves

In [ ]:
roc_data = plot_roc_curves(y_true, y_proba)
pr_data  = plot_pr_curves(y_true, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
COLORS = ["steelblue", "darkorange", "green", "red"]

for i, label in enumerate(LABEL_COLS):
    axes[0].plot(roc_data[label]["fpr"], roc_data[label]["tpr"],
                 label=f"{label} (AUC={roc_data[label]['auc']:.3f})", color=COLORS[i])
    axes[1].plot(pr_data[label]["recall"], pr_data[label]["precision"],
                 label=f"{label} (AP={pr_data[label]['ap']:.3f})", color=COLORS[i])

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curves"); axes[0].legend(fontsize=9)
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curves"); axes[1].legend(fontsize=9)
fig.suptitle("Bi-LSTM + char-CNN — Validation Set")
plt.tight_layout()
plt.show()

## 12. Error Analysis

Classifies FP/FN into the four taxonomy buckets required by the report (RQ3 & RQ4).

In [ ]:
texts = val_df["source_text"].tolist()

error_results = classify_errors(
    texts, y_true, y_pred,
    max_examples=20,
)

summary_df = error_summary(error_results)
print("Error counts by type and label:")
display(summary_df.pivot(index="error_type", columns="label", values="count").fillna(0).astype(int))

In [ ]:
def show_bucket(key: str, n: int = 5) -> None:
    records = error_results.get(key, [])
    print(f"\n{'─'*60}")
    print(f"{key}  ({len(records)} total, showing {min(n, len(records))})")
    print(f"{'─'*60}")
    for r in records[:n]:
        print(f"  [{r['label']}] {r['text_snippet'][:120].replace(chr(10), ' ')}")

show_bucket("FP_negation")
show_bucket("FP_hypothetical")
show_bucket("FN_obfuscated")
show_bucket("FN_implicit")

## 13. Save Results

In [ ]:
out_path = RESULTS_DIR / "bilstm_metrics.json"

save_results(
    metrics,
    path=out_path,
    model_name="bilstm_charcnn",
    n_samples=len(val_df),
    threshold=THRESHOLD,
    extra_metadata={
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "vocab_size": vocab.n_words,
        "glove_available": GLOVE_PATH.exists(),
        "checkpoint": str(CKPT),
    },
)

print(f"Saved to {out_path}")